In [2]:
# Standard library imports
import os
import sys
import re
import pickle

# Third-party libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import normalize
from sklearn.linear_model import LinearRegression
from sentence_transformers import SentenceTransformer

# Append parent directory to path for local imports
sys.path.append(os.path.abspath(".."))

# Local application imports
from src.utils.preprocessing import (
    load_clean_data,
    load_content_model,
    load_hybrid_model,
    preprocess_content_text,
    prepare_matrices,
    get_users_for_eval
)
from src.utils.mappings import (
    get_toeic_part_mapping,
    get_subject_categories,
    create_lecture_title
)
from src.evaluation.metrics import (
    precision_at_k,
    recall_at_k,
    rmse_score,
    calculate_precision_recall_at_k,
    calculate_rmse
)

# Set random seed and configurations
np.random.seed(42)
torch.manual_seed(42)
sns.set_theme(style="whitegrid", palette="viridis")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
# Load data
lectures_df, merged_df = load_clean_data()
print(f"Lectures: {len(lectures_df)}, Interactions: {len(merged_df)}")

# Load existing models
content_model = load_content_model('../models/content_based_model_best.pkl')

Loading from paths:
Lectures: c:\Users\karat\Downloads\Data-Driven-Personalized-Educational-Content-Recommendation-System\data\cleaned\cleaned_lectures.csv
Merged: c:\Users\karat\Downloads\Data-Driven-Personalized-Educational-Content-Recommendation-System\data\cleaned\merged_cleaned_data.csv
Successfully loaded 1021 lectures and 117167 interactions
Lectures: 1021, Interactions: 117167


In [4]:
# Load the hybrid model — now auto-initialized
hybrid_model = load_hybrid_model('../models/svd_hybrid_model_best.pkl')
recommender = hybrid_model['recommender']

In [5]:
# Use it
train_matrix, test_matrix, user_map, item_map = prepare_matrices(merged_df)
reverse_item_map = {v: k for k, v in item_map.items()}
hybrid_model['recommender'].train_matrix = train_matrix

In [6]:
# Prepare bundle features (from notebooks/02_tf_idf.ipynb)
bundle_info = merged_df.groupby('bundle_id').agg({
    'part': 'first',
    'tags': lambda x: ';'.join(set(str(i) for i in x if pd.notna(i))),
    'question_id': lambda x: len(set(x))
}).reset_index()
bundle_info.columns = ['bundle_id', 'part', 'tags', 'question_count']
part_names = get_toeic_part_mapping()
bundle_info['part_name'] = bundle_info['part'].map(part_names)
bundle_info['subject_category'] = bundle_info['tags'].apply(lambda x: get_subject_categories(x)[0] if get_subject_categories(x) else "General")
bundle_info['content_text'] = (
    bundle_info['part_name'].fillna('') + ' ' +
    bundle_info['subject_category'].fillna('') + ' ' +
    bundle_info['tags'].fillna('')
)
bundle_info['content_text'] = bundle_info['content_text'].apply(preprocess_content_text)

In [7]:
# Calculate bundle difficulty
bundle_difficulty = merged_df.groupby('bundle_id').apply(
    lambda x: (x['user_answer'] == x['correct_answer']).mean()
).reset_index()
bundle_difficulty.columns = ['bundle_id', 'success_rate']
bundle_info = bundle_info.merge(bundle_difficulty, on='bundle_id', how='left')
bundle_info['success_rate'] = bundle_info['success_rate'].fillna(0.5)

C:\Users\karat\AppData\Local\Temp\ipykernel_8476\863771296.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  bundle_difficulty = merged_df.groupby('bundle_id').apply(
